In [1]:
from google.cloud import bigquery, storage
from pyspark.sql import SparkSession
import datetime
import json

In [2]:
# Initialize Google Cloud Storage & BigQuery
storage_client = storage.Client()
bq_client = bigquery.Client()

In [3]:
spark = SparkSession.builder \
    .appName("HospitalAToLanding") \
    .config("spark.extraListeners", "") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 07:38:16 INFO SparkEnv: Registering MapOutputTracker
26/05/12 07:38:16 INFO SparkEnv: Registering BlockManagerMaster
26/05/12 07:38:16 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/12 07:38:17 INFO SparkEnv: Registering OutputCommitCoordinator


In [4]:
# GCS Configuration
GCS_BUCKET = "healthcare-bucket-minhld"
HOSPITAL_NAME = "hospital-a"
LANDING_PATH = f"gs://{GCS_BUCKET}/landing/{HOSPITAL_NAME}"
ARCHIVE_PATH = f"gs://{GCS_BUCKET}/landing/{HOSPITAL_NAME}/archive/"
CONFIG_FILE_PATH = f"gs://{GCS_BUCKET}/configs/load_config.csv"

In [5]:
#Bigquery Configuration
BQ_PROJECT = 'healthcare-496102'
BQ_AUDIT_TABLE = f"{BQ_PROJECT}.temp_dataset.audit_log"
BQ_LOG_TABLE = f"{BQ_PROJECT}.temp_dataset.pipeline_logs"
BQ_TEMP_PATH = f"{GCS_BUCKET}/temp/"

In [6]:
POSTGRES_CONFIG = {
    "url": "jdbc:postgresql://10.126.0.3:5432/hospital_a_db",
    "driver": "org.postgresql.Driver",
    "user": "minhld40",
    "password": "Leducminh1406@"
}

In [7]:
log_entries = []

def log_event(event_type, message, table=None):
    log_entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "event_type": event_type,
        "message": message,
        "table": table
    }
    log_entries.append(log_entry)
    print(f"[{log_entry['timestamp']}] {event_type} - {message}")

In [8]:
def read_config_file():
    df = spark.read.csv(CONFIG_FILE_PATH, header=True)
    log_event("INFO", "Read config file successfully")
    return df

In [9]:
config_df = read_config_file()

[2026-05-12T07:39:36.122445] INFO - Read config file successfully


In [10]:
def save_logs_to_gcs():
    """Save logs to a JSON file and upload to GCS"""
    log_filename = f"pipeline_log_{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}.json"
    log_filepath = f"temp/pipeline_logs/{log_filename}"

    # Chuyển đổi danh sách log_entries sang định dạng JSON
    json_data = json.dumps(log_entries, indent=4)

    # Lấy bucket và tạo blob (đối tượng file trên GCS)
    bucket = storage_client.bucket(GCS_BUCKET)
    blob = bucket.blob(log_filepath)

    # Upload dữ liệu JSON dưới dạng string
    blob.upload_from_string(json_data, content_type="application/json")

    print(f"Logs successfully saved to GCS at gs://{GCS_BUCKET}/{log_filepath}")

In [11]:
def save_logs_to_bigquery():
    """Save logs to BigQuery"""
    if log_entries:
        # Tạo Spark DataFrame từ danh sách log_entries
        log_df = spark.createDataFrame(log_entries)
        
        # Ghi dữ liệu vào bảng BigQuery
        log_df.write.format("bigquery") \
            .option("table", BQ_LOG_TABLE) \
            .option("temporaryGcsBucket", BQ_TEMP_PATH) \
            .mode("append") \
            .save()
            
        print("Logs stored in BigQuery for future analysis")

In [12]:
# Function to Move Existing Files to Archive
def move_existing_files_to_archive(table):
    blobs = list(storage_client.bucket(GCS_BUCKET).list_blobs(prefix=f"landing/{HOSPITAL_NAME}/{table}/"))
    existing_files = [blob.name for blob in blobs if blob.name.endswith(".json")]

    if not existing_files:
        log_event("INFO", f"No existing files for table {table}")
        return

    run_ts = datetime.datetime.now().strftime("%H%M%S%f")
    for file in existing_files:
        source_blob = storage_client.bucket(GCS_BUCKET).blob(file)

        # Extract Date from File Name
        date_part = file.split("_")[-1].split(".")[0]
        year, month, day = date_part[-4:], date_part[2:4], date_part[:2]
        filename = file.split("/")[-1]

        # Move to Archive
        archive_path = f"landing/{HOSPITAL_NAME}/archive/{table}/{year}/{month}/{day}/{run_ts}/{filename}"
        destination_blob = storage_client.bucket(GCS_BUCKET).blob(archive_path)

        # Copy file to archive and delete original
        storage_client.bucket(GCS_BUCKET).copy_blob(source_blob, storage_client.bucket(GCS_BUCKET), destination_blob.name)
        source_blob.delete()

        log_event("INFO", f"Moved {file} to {archive_path}", table=table)

In [13]:
# Function to Get Latest Watermark from BigQuery Audit Table
def get_latest_watermark(table_name):
    query = f"""
        SELECT MAX(load_timestamp) AS latest_timestamp
        FROM `{BQ_AUDIT_TABLE}`
        WHERE tablename = '{table_name}' and data_source = 'hospital_a_db'
    """
    query_job = bq_client.query(query)
    result = query_job.result()
    for row in result:
        return row.latest_timestamp if row.latest_timestamp else "1900-01-01 00:00:00"
    return "1900-01-01 00:00:00"

In [14]:
def extract_and_save_to_landing(table, load_type, watermark_col):
    try:
        clean_load_type = load_type.strip().lower()

        # Get watermark only for incremental
        if clean_load_type == "incremental":
            last_watermark = get_latest_watermark(table)
        else:
            last_watermark = None

        log_event("INFO", f"Latest watermark for {table}: {last_watermark}", table=table)

        # Build query
        if clean_load_type == "full":
            query = f"(SELECT * FROM {table}) AS t"
        else:
            query = f"""
                (SELECT * FROM {table}
                 WHERE {watermark_col} > TIMESTAMP '{last_watermark}'
                ) AS t
            """

        log_event("DEBUG", f"Query: {query}", table=table)

        # Read from Postgres
        df = (spark.read.format("jdbc")
              .option("url", POSTGRES_CONFIG["url"])
              .option("user", POSTGRES_CONFIG["user"])
              .option("password", POSTGRES_CONFIG["password"])
              .option("driver", POSTGRES_CONFIG["driver"])
              .option("dbtable", query)
              .load())

        record_count = df.count()
        log_event("INFO", f"Record count for {table}: {record_count}", table=table)

        # IMPORTANT: Skip if no data
        if record_count == 0:
            log_event("INFO", f"No new records for {table}. Skipping.", table=table)
            return 0

        # Write JSON
        today = datetime.datetime.today().strftime('%d%m%Y')
        JSON_FILE_PATH = f"landing/{HOSPITAL_NAME}/{table}/{table}_{today}.json"

        bucket = storage_client.bucket(GCS_BUCKET)
        blob = bucket.blob(JSON_FILE_PATH)

        blob.upload_from_string(
            df.toPandas().to_json(orient="records", lines=True),
            content_type="application/json"
        )

        log_event("SUCCESS", f"JSON written to gs://{GCS_BUCKET}/{JSON_FILE_PATH}", table=table)

        # Insert Audit ONLY when data exists
        audit_df = spark.createDataFrame([
            ("hospital_a_db", table, clean_load_type,
             record_count, datetime.datetime.now(), "SUCCESS")],
            ["data_source", "tablename", "load_type",
             "record_count", "load_timestamp", "status"])

        (audit_df.write.format("bigquery")
         .option("table", BQ_AUDIT_TABLE)
         .option("temporaryGcsBucket", GCS_BUCKET)
         .mode("append")
         .save())

        log_event("SUCCESS", f"Audit log updated for {table}", table=table)

        return record_count

    except Exception as e:
        log_event("ERROR", f"Error processing {table}: {str(e)}", table=table)
        return 0

In [15]:
for row in config_df.collect():
    if row["is_active"] == '1' and row["datasource"] == "hospital_a_db":
        db, src, table, load_type, watermark, _, targetpath = row
        move_existing_files_to_archive(table)
        extract_and_save_to_landing(table, load_type, watermark)

[2026-05-12T07:39:53.333434] INFO - No existing files for table encounters
[2026-05-12T07:39:54.426576] INFO - Latest watermark for encounters: 1900-01-01 00:00:00
[2026-05-12T07:39:54.426664] DEBUG - Query: 
                (SELECT * FROM encounters
                 WHERE ModifiedDate > TIMESTAMP '1900-01-01 00:00:00'
                ) AS t
            
[2026-05-12T07:39:56.301772] INFO - Record count for encounters: 10000


[2026-05-12T07:39:58.099731] SUCCESS - JSON written to gs://healthcare-bucket-minhld/landing/hospital-a/encounters/encounters_12052026.json


[2026-05-12T07:40:09.516888] SUCCESS - Audit log updated for encounters
[2026-05-12T07:40:09.548837] INFO - No existing files for table patients
[2026-05-12T07:40:10.395309] INFO - Latest watermark for patients: 1900-01-01 00:00:00
[2026-05-12T07:40:10.395395] DEBUG - Query: 
                (SELECT * FROM patients
                 WHERE ModifiedDate > TIMESTAMP '1900-01-01 00:00:00'
                ) AS t
            


[2026-05-12T07:40:12.396309] INFO - Record count for patients: 5000
[2026-05-12T07:40:12.975008] SUCCESS - JSON written to gs://healthcare-bucket-minhld/landing/hospital-a/patients/patients_12052026.json


[2026-05-12T07:40:26.876646] SUCCESS - Audit log updated for patients
[2026-05-12T07:40:26.907065] INFO - No existing files for table transactions
[2026-05-12T07:40:27.687130] INFO - Latest watermark for transactions: 1900-01-01 00:00:00
[2026-05-12T07:40:27.687218] DEBUG - Query: 
                (SELECT * FROM transactions
                 WHERE ModifiedDate > TIMESTAMP '1900-01-01 00:00:00'
                ) AS t
            
[2026-05-12T07:40:28.122707] INFO - Record count for transactions: 10000
[2026-05-12T07:40:29.414638] SUCCESS - JSON written to gs://healthcare-bucket-minhld/landing/hospital-a/transactions/transactions_12052026.json


[2026-05-12T07:40:35.930062] SUCCESS - Audit log updated for transactions
[2026-05-12T07:40:35.955895] INFO - No existing files for table providers
[2026-05-12T07:40:35.955953] INFO - Latest watermark for providers: None
[2026-05-12T07:40:35.955964] DEBUG - Query: (SELECT * FROM providers) AS t
[2026-05-12T07:40:36.243415] INFO - Record count for providers: 25
[2026-05-12T07:40:36.633595] SUCCESS - JSON written to gs://healthcare-bucket-minhld/landing/hospital-a/providers/providers_12052026.json


[2026-05-12T07:40:43.202774] SUCCESS - Audit log updated for providers
[2026-05-12T07:40:43.237982] INFO - No existing files for table departments
[2026-05-12T07:40:43.238033] INFO - Latest watermark for departments: None
[2026-05-12T07:40:43.238041] DEBUG - Query: (SELECT * FROM departments) AS t
[2026-05-12T07:40:43.500901] INFO - Record count for departments: 20
[2026-05-12T07:40:43.881872] SUCCESS - JSON written to gs://healthcare-bucket-minhld/landing/hospital-a/departments/departments_12052026.json


[2026-05-12T07:40:51.998709] SUCCESS - Audit log updated for departments


In [16]:
for row in config_df.collect():
    if row["is_active"] == '1' and row["datasource"] == "hospital_a_db":
        db, src, table, load_type, watermark, _, targetpath = row
        record_count = extract_and_save_to_landing(table, load_type, watermark)
        # Chỉ archive khi incremental và có data mới
        if load_type.lower() == "incremental" and record_count > 0:
            move_existing_files_to_archive(table)

[2026-05-12T07:41:38.080690] INFO - Latest watermark for encounters: 2026-05-12 07:39:58.099826+00:00
[2026-05-12T07:41:38.080776] DEBUG - Query: 
                (SELECT * FROM encounters
                 WHERE ModifiedDate > TIMESTAMP '2026-05-12 07:39:58.099826+00:00'
                ) AS t
            
[2026-05-12T07:41:38.418207] INFO - Record count for encounters: 0
[2026-05-12T07:41:38.418340] INFO - No new records for encounters. Skipping.
[2026-05-12T07:41:39.223032] INFO - Latest watermark for patients: 2026-05-12 07:40:12.975346+00:00
[2026-05-12T07:41:39.223112] DEBUG - Query: 
                (SELECT * FROM patients
                 WHERE ModifiedDate > TIMESTAMP '2026-05-12 07:40:12.975346+00:00'
                ) AS t
            
[2026-05-12T07:41:39.624680] INFO - Record count for patients: 0
[2026-05-12T07:41:39.624801] INFO - No new records for patients. Skipping.
[2026-05-12T07:41:40.229324] INFO - Latest watermark for transactions: 2026-05-12 07:40:29.414732+00:00


[2026-05-12T07:41:46.921126] SUCCESS - Audit log updated for providers
[2026-05-12T07:41:46.921290] INFO - Latest watermark for departments: None
[2026-05-12T07:41:46.921304] DEBUG - Query: (SELECT * FROM departments) AS t
[2026-05-12T07:41:47.179410] INFO - Record count for departments: 20
[2026-05-12T07:41:47.424300] SUCCESS - JSON written to gs://healthcare-bucket-minhld/landing/hospital-a/departments/departments_12052026.json


[2026-05-12T07:42:00.619969] SUCCESS - Audit log updated for departments


In [17]:
save_logs_to_gcs()

Logs successfully saved to GCS at gs://healthcare-bucket-minhld/temp/pipeline_logs/pipeline_log_20260512074257.json


In [18]:
save_logs_to_bigquery()

Logs stored in BigQuery for future analysis


26/05/12 07:51:26 ERROR YarnClientSchedulerBackend: YARN application has exited unexpectedly with state KILLED! Check the YARN application logs for more details.
26/05/12 07:51:26 ERROR YarnClientSchedulerBackend: Diagnostics message: Application application_1778569936395_0002 was killed by user ducmi at 10.148.0.5
26/05/12 07:51:26 ERROR ApplicationMaster: Exception from Reporter thread.
org.apache.hadoop.yarn.exceptions.ApplicationAttemptNotFoundException: Application attempt appattempt_1778569936395_0002_000001 doesn't exist in ApplicationMasterService cache.
	at org.apache.hadoop.yarn.server.resourcemanager.ApplicationMasterService.allocate(ApplicationMasterService.java:408)
	at org.apache.hadoop.yarn.api.impl.pb.service.ApplicationMasterProtocolPBServiceImpl.allocate(ApplicationMasterProtocolPBServiceImpl.java:60)
	at org.apache.hadoop.yarn.proto.ApplicationMasterProtocol$ApplicationMasterProtocolService$2.callBlockingMethod(ApplicationMasterProtocol.java:106)
	at org.apache.hadoo